# Section C: Machine learning

**Pair C:** Njihia Muranga and Gakuo Philip Ndirangu

How conserved is the Ebola genome across *Ebolavirus* species, and can genome sequence alone separate Bundibugyo from Zaire and Sudan?

This notebook covers stage 6 of the project.

* **Needs:** `handoff_A.pkl` from Section A
* **Saves:** `handoff_C.pkl`, the model result tables and the stage 6 figures

The four section notebooks are run in order: A, then B and C (either order), then D. Run them in the same Colab session, or upload the earlier `handoff_*.pkl` files into `ebola_project/outputs/` first. Keep the settings cell the same in all four notebooks, with your email in `NCBI_EMAIL`.

## Setup

Installs the tools the notebook needs (Biopython, MAFFT, FastTree) and loads the Python libraries.

In [ ]:
import os, sys, re, io, json, time, gzip, shutil, subprocess, pickle, hashlib, zipfile, warnings, random, contextlib, itertools
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
sys.setrecursionlimit(20000)

IN_COLAB = "google.colab" in sys.modules

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{r.stderr[-2000:]}")
    return r

try:
    import Bio
except ImportError:
    sh("pip -q install biopython")
    import Bio
if not (shutil.which("mafft") and (shutil.which("fasttree") or shutil.which("FastTree"))):
    print("Installing MAFFT and FastTree (about 1 minute)...")
    sh("apt-get -qq update && apt-get -qq install -y mafft fasttree", check=False)
print("MAFFT:", shutil.which("mafft"), "| FastTree:", shutil.which("fasttree") or shutil.which("FastTree"))
print("CPU cores:", os.cpu_count(), "| Biopython", Bio.__version__)

The settings for the whole run. Put your own email in `NCBI_EMAIL` (NCBI asks for it when downloading). The other settings can stay as they are.

In [ ]:
NCBI_EMAIL = "your_email@example.com"  # put your own email here
NCBI_API_KEY = None
DOWNLOAD_METHOD = "datasets"
ENTREZ_MAX_PER_SPECIES = 2500

MIN_LEN, MAX_LEN = 17000, 20500
MAX_N_FRACTION = 0.01  # drop genomes with more than 1% unknown bases
ZAIRE_CAP_PER_OUTBREAK = 40  # keep at most this many Zaire genomes per outbreak
RANDOM_SEED = 42

EDA_KMER_K = 4
KMER_K = 6  # k-mer length used by the classifiers
ML_CLASSES = ["Bundibugyo", "Sudan", "Zaire"]
FRAGMENT_TEST = True
FRAGMENT_LENGTHS = [250, 500, 1000, 2500]
N_JOBS = -1

ALIGN_MAX_SEQS = 400  # align at most this many genomes
MAFFT_MODE = "fast"  # "fast" or "auto" (slower, more accurate)
TREE_GAMMA = False
REUSE_CACHE = True

WORKDIR = Path("ebola_project")
DATA_DIR, REF_DIR, FIG_DIR, OUT_DIR = [WORKDIR / d for d in ("data", "data/refs", "figures", "outputs")]
for d in (DATA_DIR, REF_DIR, FIG_DIR, OUT_DIR): d.mkdir(parents=True, exist_ok=True)

SPECIES = ["Zaire", "Sudan", "Bundibugyo", "Reston", "Tai Forest", "Bombali"]
SPECIES_QUERY = {
    "Zaire": ["Orthoebolavirus zairense", "Zaire ebolavirus"],
    "Sudan": ["Orthoebolavirus sudanense", "Sudan ebolavirus"],
    "Bundibugyo": ["Orthoebolavirus bundibugyoense", "Bundibugyo ebolavirus"],
    "Reston": ["Orthoebolavirus restonense", "Reston ebolavirus"],
    "Tai Forest": ["Orthoebolavirus taiense", "Tai Forest ebolavirus"],
    "Bombali": ["Orthoebolavirus bombaliense", "Bombali ebolavirus"],
}
REF_ACC = {"Zaire": "NC_002549.1", "Sudan": "NC_006432.1", "Bundibugyo": "NC_014373.1",
           "Reston": "NC_004161.1", "Tai Forest": "NC_014372.1", "Bombali": "NC_039345.1"}
GENE_ORDER = ["NP", "VP35", "VP40", "GP", "VP30", "VP24", "L"]
COLORS = {"Zaire": "#1f77b4", "Sudan": "#d62728", "Bundibugyo": "#2ca02c",
          "Reston": "#9467bd", "Tai Forest": "#ff7f0e", "Bombali": "#8c564b"}

TIMINGS, RESULTS = [], {}
@contextlib.contextmanager
def timed(label):
    t = time.time(); print(f"> {label} ...")
    try: yield
    finally:
        dt = time.time() - t; TIMINGS.append((label, dt)); print(f"< {label}: {dt/60:.1f} min ({dt:.0f} s)")

def savefig(name):
    plt.tight_layout(); plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight"); plt.show()

sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(RANDOM_SEED); random.seed(RANDOM_SEED)

if "example.com" in NCBI_EMAIL:
    raise ValueError("Set NCBI_EMAIL (above) to your real email address before running the real download.")

Helper functions for reading and writing FASTA files, finding the genes in a reference genome and downloading data from NCBI.

In [ ]:
def read_fasta(path):
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rt") as fh:
        name, desc, buf = None, "", []
        for line in fh:
            if line.startswith(">"):
                if name is not None: yield name, desc, "".join(buf).upper()
                head = line[1:].strip().split(None, 1)
                name, desc, buf = head[0], (head[1] if len(head) > 1 else ""), []
            else: buf.append(line.strip())
        if name is not None: yield name, desc, "".join(buf).upper()

def write_fasta(path, items, width=80):
    with open(path, "w") as fh:
        for name, seq in items:
            fh.write(f">{name}\n")
            for i in range(0, len(seq), width): fh.write(seq[i:i+width] + "\n")

def base_acc(a): return a.split(".")[0]

def canon_gene(q):
    g = (q.get("gene", [""])[0]).strip(); prod = (q.get("product", [""])[0]).lower()
    if g.upper() in GENE_ORDER: return g.upper()
    if "nucleoprotein" in prod: return "NP"
    if "vp35" in prod or "cofactor" in prod: return "VP35"
    if "vp40" in prod or "matrix" in prod: return "VP40"
    if "vp30" in prod: return "VP30"
    if "vp24" in prod: return "VP24"
    if "glycoprotein" in prod and "secreted" not in prod and "small" not in prod: return "GP"
    if "polymerase" in prod or prod.strip() == "l": return "L"
    return None

def gene_table(rec):
    spans = {}
    for f in rec.features:
        if f.type != "CDS": continue
        g = canon_gene(f.qualifiers)
        if g is None: continue
        s, e = int(f.location.start) + 1, int(f.location.end)
        a, b = spans.get(g, (s, e)); spans[g] = (min(a, s), max(b, e))
    df = pd.DataFrame([(g, *spans[g]) for g in GENE_ORDER if g in spans], columns=["gene", "start", "end"])
    missing = [g for g in GENE_ORDER if g not in spans]
    if missing: print(f"  WARNING {rec.id}: genes not found in annotation: {missing}")
    return df

def fetch_refs():
    from Bio import Entrez, SeqIO
    Entrez.email = NCBI_EMAIL
    if NCBI_API_KEY: Entrez.api_key = NCBI_API_KEY
    refs, genes = {}, {}
    for sp, acc in REF_ACC.items():
        p = REF_DIR / f"{base_acc(acc)}.gb"
        if not p.exists():
            with Entrez.efetch(db="nuccore", id=base_acc(acc), rettype="gb", retmode="text") as h: p.write_text(h.read())
            time.sleep(0.4)
        rec = SeqIO.read(p, "genbank")
        refs[sp] = str(rec.seq).upper(); genes[sp] = gene_table(rec)
        REF_ACC[sp] = rec.id
    return refs, genes

def jget(d, *keys):
    for k in keys:
        if not isinstance(d, dict): return None
        d = d.get(k)
    return d

def download_datasets_cli():
    ds = DATA_DIR / "datasets"
    if not ds.exists():
        sh(f"curl -sSL -o {ds} https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets && chmod +x {ds}")
    records, rows = {}, []
    for sp, names in SPECIES_QUERY.items():
        got = False
        for nm in names:
            z = DATA_DIR / f"{sp.replace(' ', '_')}.zip"
            r = sh(f'{ds} download virus genome taxon "{nm}" --complete-only --filename {z}', check=False)
            if r.returncode != 0 or not z.exists(): continue
            out = DATA_DIR / f"unz_{sp.replace(' ', '_')}"; shutil.rmtree(out, ignore_errors=True)
            with zipfile.ZipFile(z) as zf: zf.extractall(out)
            fna = next(out.rglob("genomic.fna"), None); rep = next(out.rglob("data_report.jsonl"), None)
            if fna is None: continue
            seqs = {n: (d, s) for n, d, s in read_fasta(fna)}
            reports = {}
            if rep is not None:
                for line in open(rep):
                    j = json.loads(line); reports[j.get("accession")] = j
            for acc, (desc, s) in seqs.items():
                j = reports.get(acc, {})
                rows.append(dict(accession=acc, species=sp, isolate=jget(j, "isolate", "name") or desc,
                                 collection_date=jget(j, "isolate", "collectionDate"),
                                 geo=jget(j, "location", "geographicLocation"), region=jget(j, "location", "geographicRegion"),
                                 desc=desc))
                records[acc] = s
            print(f"  {sp}: {len(seqs)} genomes (query '{nm}')"); got = True; break
        if not got: print(f"  WARNING: nothing downloaded for {sp}")
    return records, pd.DataFrame(rows)

def download_entrez():
    from Bio import Entrez, SeqIO
    Entrez.email = NCBI_EMAIL
    if NCBI_API_KEY: Entrez.api_key = NCBI_API_KEY
    records, rows = {}, []
    for sp, names in SPECIES_QUERY.items():
        for nm in names:
            term = f'"{nm}"[Organism] AND {MIN_LEN}:{MAX_LEN}[SLEN]'
            res = Entrez.read(Entrez.esearch(db="nuccore", term=term, retmax=0, usehistory="y"))
            n = int(res["Count"])
            if n > 0: break
        n = min(n, ENTREZ_MAX_PER_SPECIES); print(f"  {sp}: fetching {n} records")
        for start in range(0, n, 200):
            for attempt in range(4):
                try:
                    h = Entrez.efetch(db="nuccore", rettype="gb", retmode="text", retstart=start, retmax=min(200, n - start),
                                      webenv=res["WebEnv"], query_key=res["QueryKey"])
                    batch = list(SeqIO.parse(h, "genbank")); break
                except Exception as ex:
                    print("   retry", attempt, ex); time.sleep(3); batch = []
            for rec in batch:
                src = next((f for f in rec.features if f.type == "source"), None); q = src.qualifiers if src else {}
                records[rec.id] = str(rec.seq).upper()
                rows.append(dict(accession=rec.id, species=sp, isolate=(q.get("isolate") or q.get("strain") or [""])[0],
                                 collection_date=(q.get("collection_date") or [None])[0], geo=(q.get("country") or [None])[0],
                                 region=None, desc=rec.description))
            time.sleep(0.4)
    return records, pd.DataFrame(rows)

Functions for counting k-mers.

In [ ]:
_LUT = np.full(256, -1, dtype=np.int64)
for _i, _c in enumerate("ACGT"): _LUT[ord(_c)] = _i

def kmer_index(seq, k):
    a = _LUT[np.frombuffer(seq.encode(), dtype=np.uint8)]
    n = len(a) - k + 1
    if n <= 0: return np.zeros(0, dtype=np.int64)
    idx = np.zeros(n, dtype=np.int64); bad = np.zeros(n, dtype=bool)
    for j in range(k):
        s = a[j:j+n]; bad |= s < 0; idx = idx * 4 + s
    idx[bad] = -1
    return idx

def kmer_freq(seq, k):
    idx = kmer_index(seq, k); idx = idx[idx >= 0]
    c = np.bincount(idx, minlength=4**k).astype(np.float32)
    return c / max(c.sum(), 1)

def kmer_matrix(seqs, k):
    return np.vstack([kmer_freq(s, k) for s in seqs])

def kmer_labels(k): return ["".join(p) for p in itertools.product("ACGT", repeat=k)]

from sklearn.preprocessing import StandardScaler

Loads the cleaned genomes and reference genomes saved by Section A.

In [ ]:
_hA = pickle.load(open(OUT_DIR / "handoff_A.pkl", "rb"))
qc, SEQ, REFS, GENES = _hA["qc"], _hA["SEQ"], _hA["REFS"], _hA["GENES"]
REF_ACC.update(_hA["REF_ACC"])
sp_of = dict(zip(qc.accession, qc.species)); ob_of = dict(zip(qc.accession, qc.outbreak))
print(f"Loaded Section A results: {len(qc)} genomes after quality control")
display(qc.species.value_counts().rename("genomes").to_frame().T)

## 6. Machine learning

Predicts the species (Bundibugyo, Sudan or Zaire) from the genome alone, using counts of short DNA words (k-mers) as features. Three models are compared: a baseline that always answers the most common species, logistic regression and a random forest. Each outbreak is held out in turn, so the model is tested on an outbreak it has never seen.

Builds the k-mer features, then trains and tests the three models. It prints accuracy and per-species recall for each model.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold, GroupKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score, confusion_matrix

ml = qc[qc.species.isin(ML_CLASSES)].reset_index(drop=True)
y = ml.species.values; groups = ml.outbreak.values
print(ml.species.value_counts().to_string(), f"\n{ml.outbreak.nunique()} outbreak groups")

with timed("Stage 6a: k-mer features"):
    X = kmer_matrix([SEQ[a] for a in ml.accession], KMER_K); print("Feature matrix:", X.shape)

MODELS = {
    "Majority baseline": lambda: DummyClassifier(strategy="most_frequent"),
    "Logistic regression": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, C=0.05, class_weight="balanced")),
    "Random forest": lambda: RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample", n_jobs=N_JOBS, random_state=RANDOM_SEED),
}

def run_cv(model_fn, splits):
    pred = np.full(len(y), None, dtype=object); skipped = 0
    for tr, te in splits:
        if not set(y[te]) <= set(y[tr]): skipped += len(te); continue
        m = model_fn(); m.fit(X[tr], y[tr]); pred[te] = m.predict(X[te])
    return pred, skipped

def metrics(pred):
    ok = pred != None; yt, yp = y[ok], pred[ok].astype(str)
    out = dict(n_evaluated=int(ok.sum()), accuracy=accuracy_score(yt, yp), balanced_accuracy=balanced_accuracy_score(yt, yp),
               macro_f1=f1_score(yt, yp, average="macro", zero_division=0))
    for c in ML_CLASSES: out[f"recall_{c}"] = recall_score(yt == c, yp == c, zero_division=0) if (yt == c).any() else np.nan
    return out

looo = list(LeaveOneGroupOut().split(X, y, groups))
print(f"{len(looo)} leave-one-outbreak-out folds")
res_rows, preds = [], {}
for name, fn in MODELS.items():
    with timed(f"Stage 6b: LOOO - {name}"):
        p, sk = run_cv(fn, looo); preds[name] = p; r = metrics(p); res_rows.append(dict(model=name, validation="Leave-one-outbreak-out", skipped=sk, **r))
min_class = pd.Series(y).value_counts().min()
if min_class >= 2:
    skf = list(StratifiedKFold(n_splits=min(5, min_class), shuffle=True, random_state=RANDOM_SEED).split(X, y))
    for name, fn in MODELS.items():
        with timed(f"Stage 6b: stratified CV - {name}"):
            p, sk = run_cv(fn, skf); res_rows.append(dict(model=name, validation=f"Stratified {len(skf)}-fold (random)", skipped=sk, **metrics(p)))
results = pd.DataFrame(res_rows); results.to_csv(OUT_DIR / "ml_results.csv", index=False)
display(results.round(3))
RESULTS["ml_results"] = results.round(4).to_dict("records")

Draws a confusion matrix for each model, and shows how the random forest did on each held-out outbreak, including any genomes it got wrong.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
for a, name in zip(ax, MODELS):
    p = preds[name]; ok = p != None
    cm = confusion_matrix(y[ok], p[ok].astype(str), labels=ML_CLASSES)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=ML_CLASSES, yticklabels=ML_CLASSES, cbar=False, ax=a)
    a.set_title(f"{name}\n(leave-one-outbreak-out)"); a.set_xlabel("predicted"); a.set_ylabel("true")
savefig("06_confusion_matrices")

p = preds["Random forest"]; ok = p != None
per_ob = (pd.DataFrame({"outbreak": groups[ok], "species": y[ok], "correct": (p[ok] == y[ok])})
          .groupby(["species", "outbreak"]).agg(genomes=("correct", "size"), accuracy=("correct", "mean")).reset_index())
print("Random forest accuracy on each held-out outbreak (worst first):"); display(per_ob.sort_values("accuracy").head(15).round(3))
per_ob.to_csv(OUT_DIR / "ml_per_outbreak_rf.csv", index=False)
mis = ml[ok & (p != y)][["accession", "species", "outbreak"]].assign(predicted=p[ok & (p != y)])
print(f"Misclassified genomes (random forest, LOOO): {len(mis)}"); display(mis.head(20))

Finds the k-mers that matter most to the random forest and maps them back onto the genes. It then tests how well the model works when it only sees a short fragment of the genome.

In [ ]:
with timed("Stage 6c: feature importance"):
    rf = RandomForestClassifier(n_estimators=500, class_weight="balanced_subsample", n_jobs=N_JOBS, random_state=RANDOM_SEED).fit(X, y)
    imp = rf.feature_importances_; labels = kmer_labels(KMER_K)
    top = pd.DataFrame({"kmer": labels, "importance": imp}).sort_values("importance", ascending=False).head(25)
    plt.figure(figsize=(8, 6)); sns.barplot(data=top, y="kmer", x="importance", color="#4c72b0"); plt.title(f"Top 25 {KMER_K}-mers (random forest importance)"); savefig("06_top_kmers")

    def importance_profile(seq, imp, k):
        idx = kmer_index(seq, k); v = np.where(idx >= 0, imp[np.maximum(idx, 0)], 0.0)
        prof = np.zeros(len(seq))
        for j in range(k): prof[j:j + len(v)] += v / k
        return prof
    imp_rows, PROF = [], {}
    for sp in ML_CLASSES:
        prof = importance_profile(REFS[sp], imp, KMER_K); PROF[sp] = prof; tot = prof.sum()
        for g in GENES[sp].itertuples():
            share = prof[g.start - 1:g.end].sum() / tot; size = (g.end - g.start + 1) / len(REFS[sp])
            imp_rows.append(dict(reference=sp, gene=g.gene, importance_share=share, enrichment=share / size))
    gene_imp = pd.DataFrame(imp_rows); gene_imp.to_csv(OUT_DIR / "gene_importance.csv", index=False)
    piv = gene_imp.pivot(index="gene", columns="reference", values="enrichment").reindex(GENE_ORDER)
    plt.figure(figsize=(7, 4.5)); sns.heatmap(piv, annot=True, fmt=".2f", cmap="YlOrRd", center=1, cbar_kws={"label": "importance enrichment (1 = proportional to length)"})
    plt.title("Where classifier-relevant k-mers fall (mapped on each reference)"); savefig("06_gene_importance")
display(piv.round(2))

if FRAGMENT_TEST:
    with timed("Stage 6d: short-fragment test"):
        FK, N_TR, N_TE = 5, 4, 4
        frng = np.random.default_rng(RANDOM_SEED)
        def frags(accs, L, n):
            Xs, ys, gs = [], [], []
            for a in accs:
                s = SEQ[a]
                for st in frng.integers(0, len(s) - L, n): Xs.append(kmer_freq(s[st:st + L], FK)); ys.append(sp_of[a]); gs.append(ob_of[a])
            return np.vstack(Xs), np.array(ys), np.array(gs)
        n_splits = min(5, ml.outbreak.nunique()); frows = []
        for L in FRAGMENT_LENGTHS:
            yt_all, yp_all = [], []
            for tr, te in GroupKFold(n_splits=n_splits).split(X, y, groups):
                if not set(y[te]) <= set(y[tr]): continue
                Xtr, ytr, _ = frags(ml.accession.values[tr], L, N_TR); Xte, yte, _ = frags(ml.accession.values[te], L, N_TE)
                m = RandomForestClassifier(n_estimators=150, class_weight="balanced_subsample", n_jobs=N_JOBS, random_state=RANDOM_SEED).fit(Xtr, ytr)
                yt_all.append(yte); yp_all.append(m.predict(Xte))
            yt_all, yp_all = np.concatenate(yt_all), np.concatenate(yp_all)
            frows.append(dict(fragment_nt=L, accuracy=accuracy_score(yt_all, yp_all), balanced_accuracy=balanced_accuracy_score(yt_all, yp_all),
                              recall_Bundibugyo=recall_score(yt_all == "Bundibugyo", yp_all == "Bundibugyo", zero_division=0)))
        frag = pd.DataFrame(frows); frag.to_csv(OUT_DIR / "ml_fragments.csv", index=False)
    display(frag.round(3)); RESULTS["fragments"] = frag.round(4).to_dict("records")
    plt.figure(figsize=(6.5, 4)); plt.plot(frag.fragment_nt, frag.balanced_accuracy, "o-", label="balanced accuracy"); plt.plot(frag.fragment_nt, frag.recall_Bundibugyo, "s--", label="Bundibugyo recall")
    plt.xscale("log"); plt.ylim(0, 1.02); plt.xlabel("fragment length (nt)"); plt.title("Classification from short fragments (outbreak-grouped CV)"); plt.legend(); savefig("06_fragments")

Saves the results so that Section D can use them.

In [ ]:
pickle.dump(dict(results=results, piv=piv, PROF=PROF, RESULTS=RESULTS, TIMINGS=TIMINGS), open(OUT_DIR / "handoff_C.pkl", "wb"))
print("Saved", OUT_DIR / "handoff_C.pkl")